# 2. Train the models

Input: `data/features.csv` from notebook 1. Output: `results/predictions.csv`,
`results/metrics.csv`, `results/importance.csv`.

**Main model:** A small neural network (implemented in PyTorch). Five sets are trained using different random seeds, and the final prediction is the average of the five; the degree of dispersion (standard deviation) among these five results serves as an indicator of the model's certainty.

**Baselines:** predict the mean gap, linear regression, random forest.

**Two feature sets:** composition only (13) vs. composition with neighbours (17).

**Evaluation:** 5-fold cross-validation, grouped by chemical formula. Different structures (polymorphs) of the same compound are always kept in the same fold; otherwise, the model might encounter nearly identical materials during training, leading to exaggerated test scores.


In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

os.makedirs("results", exist_ok=True)

data = pd.read_csv("data/features.csv")
COMP = ["X_mean", "X_diff", "radius_mean", "radius_diff", "IE_mean", "IE_diff",
        "row_mean", "row_diff", "group_mean", "group_diff", "s_val", "p_val", "d_val"]
PAIR = ["nn_dist", "coord", "bond_ratio", "unlike_frac"]
FEATURE_SETS = {"composition": COMP, "composition+neighbours": COMP + PAIR}

y = data.band_gap.values
folds = list(GroupKFold(n_splits=5).split(data, y, groups=data.formula))
print(len(data), "materials,", data.formula.nunique(), "formulas")

2307 materials, 1840 formulas


## Neural network

The neural network we used consists of three layers with 128, 64, and 32 neurons, respectively, and outputs the predicted band gap.

To prevent overfitting during training, we used two techniques:

Randomly deactivating a portion of the neurons.
Reserving 10% of the data for validation and stopping the training process once the error ceases to decrease .

We tested several different network architectures, and this configuration was among the best, so we selected it.
(on an earlier download of the data (2,314 materials))

| hidden layers | dropout | MAE (eV) |
|---|---|---|
| 64-32 | 0 | 0.450 |
| 128-64 | 0 | 0.443 |
| 128-64 | 0.2 | 0.437 |
| **128-64-32** | **0.1** | **0.423** |
| 256-128-64 | 0.2 | 0.424 |


In [2]:
def train_nn(X, y, seed, epochs=1000, patience=50):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    # keep 10% of the training data for early stopping
    idx = rng.permutation(len(X))
    n_val = len(X) // 10
    X_val, y_val = torch.tensor(X[idx[:n_val]], dtype=torch.float32), torch.tensor(y[idx[:n_val]], dtype=torch.float32)
    X_tr, y_tr = torch.tensor(X[idx[n_val:]], dtype=torch.float32), torch.tensor(y[idx[n_val:]], dtype=torch.float32)

    model = nn.Sequential(nn.Linear(X.shape[1], 128), nn.ReLU(), nn.Dropout(0.1),
                          nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.1),
                          nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
                          nn.Linear(32, 1))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    best_loss, best_state, wait = np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr))
        for start in range(0, len(X_tr), 64):
            batch = perm[start:start + 64]
            optimizer.zero_grad()
            loss = loss_fn(model(X_tr[batch]).squeeze(), y_tr[batch])
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val).squeeze(), y_val).item()
        if val_loss < best_loss:
            best_loss, wait = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait > patience:
                break
    model.load_state_dict(best_state)
    return model


def predict_nn(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32)).squeeze().numpy()

## Cross-validation

We used 5-fold cross-validation to train and test all models.

To determine which features the neural network relies on most, we shuffled one feature at a time  and observed the resulting increase in error. A larger increase in error indicates that the feature is more important.

In [3]:
N_SEEDS = 5
pred = data[["material_id", "formula", "band_gap"]].copy()
pred["fold"] = -1
importance = []

for k, (train, test) in enumerate(folds):
    pred.loc[test, "fold"] = k
    pred.loc[test, "mean_baseline"] = y[train].mean()

    for name, cols in FEATURE_SETS.items():
        scaler = StandardScaler().fit(data.loc[train, cols])
        X_train = scaler.transform(data.loc[train, cols])
        X_test = scaler.transform(data.loc[test, cols])

        lin = LinearRegression().fit(X_train, y[train])
        pred.loc[test, f"linear_{name}"] = lin.predict(X_test)

        rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, max_features=0.33, random_state=0, n_jobs=-1)
        rf.fit(X_train, y[train])
        pred.loc[test, f"rf_{name}"] = rf.predict(X_test)

        models = [train_nn(X_train, y[train], seed) for seed in range(N_SEEDS)]
        p_all = np.array([predict_nn(m, X_test) for m in models])
        pred.loc[test, f"nn_{name}"] = p_all.mean(axis=0)
        pred.loc[test, f"nn_std_{name}"] = p_all.std(axis=0)

        if name == "composition+neighbours":
            base_mae = np.abs(p_all.mean(axis=0) - y[test]).mean()
            rng = np.random.default_rng(k)
            for j, col in enumerate(cols):
                X_shuffled = X_test.copy()
                X_shuffled[:, j] = rng.permutation(X_shuffled[:, j])
                p = np.mean([predict_nn(m, X_shuffled) for m in models], axis=0)
                importance.append({"fold": k, "feature": col, "mae_increase": np.abs(p - y[test]).mean() - base_mae})

    print(f"fold {k} done")

pred.to_csv("results/predictions.csv", index=False)
pd.DataFrame(importance).to_csv("results/importance.csv", index=False)

fold 0 done
fold 1 done
fold 2 done
fold 3 done
fold 4 done


## Errors

MAE = mean absolute error (eV), RMSE = root mean squared error (eV), R² = fraction of
the variance explained. Mean and standard deviation over the 5 folds.

In [4]:
rows = []
model_cols = ["mean_baseline"] + [c for c in pred.columns if c.startswith(("linear_", "rf_", "nn_composition"))]
for col in model_cols:
    for k in range(5):
        f = pred[pred.fold == k]
        err = f[col] - f.band_gap
        rows.append({"model": col, "fold": k,
                     "MAE": err.abs().mean(),
                     "RMSE": np.sqrt((err ** 2).mean()),
                     "R2": 1 - (err ** 2).sum() / ((f.band_gap - f.band_gap.mean()) ** 2).sum()})

metrics = pd.DataFrame(rows).groupby("model")[["MAE", "RMSE", "R2"]].agg(["mean", "std"])
metrics = metrics.sort_values(("MAE", "mean"))
metrics.to_csv("results/metrics.csv")
metrics.round(3)

MAE          RMSE            R2       
                                mean    std   mean    std   mean    std
model                                                                  
nn_composition+neighbours      0.409  0.020  0.536  0.022  0.515  0.036
rf_composition+neighbours      0.438  0.013  0.553  0.019  0.483  0.032
nn_composition                 0.445  0.017  0.574  0.019  0.443  0.038
rf_composition                 0.448  0.015  0.568  0.018  0.456  0.031
linear_composition+neighbours  0.554  0.016  0.677  0.013  0.227  0.023
linear_composition             0.564  0.012  0.683  0.011  0.214  0.023
mean_baseline                  0.656  0.006  0.771  0.005 -0.004  0.006